# RAILGUN+ — Step 1: Corrector Variant Comparison



## 1. Setup 

In [ ]:
import os, sys, pickle
!git clone -q https://github.com/tay805/railgun-plus.git /kaggle/working/railgun-plus
!pip install -q pogema
sys.path.insert(0,'/kaggle/working/railgun-plus/src')
import torch

CKPT='/kaggle/input/your-checkpoints/best.pt'
LACAM_BIN='/kaggle/input/lacam-binary/main'  # set None if unavailable
if LACAM_BIN and os.path.exists(LACAM_BIN): !chmod +x {LACAM_BIN}
else: LACAM_BIN=None
RESULTS_DIR='/kaggle/working/results_step1'
os.makedirs(RESULTS_DIR, exist_ok=True)

## 2. Load model

In [ ]:
from railgun_plus.models import RailgunUNet
device='cuda' if torch.cuda.is_available() else 'cpu'
model=RailgunUNet(6,5,base=64).to(device)
ck=torch.load(CKPT,map_location=device)
model.load_state_dict(ck['model']);model.eval()
print('loaded epoch',ck.get('epoch'),'on',device)

## 3. Reuse the SAME test sets from Notebook 02
This is important — same trained model AND same test instances, so the comparison is apples-to-apples.

In [ ]:
from railgun_plus.data.generate import generate_with_pogema
AGENT_COUNTS=[16,32,64,96,128]
N_PER=50
# load the test_sets.pkl from your previous eval run if you have it
PREV='/kaggle/working/results/test_sets.pkl'
if os.path.exists(PREV):
    test_sets=pickle.load(open(PREV,'rb'))
    print('reused', {k:len(v) for k,v in test_sets.items()})
else:
    test_sets={}
    for k in AGENT_COUNTS:
        test_sets[k]=generate_with_pogema(N_PER,32,0.2,k,seed=1000+k)
        print(k,'agents:',len(test_sets[k]))
    pickle.dump(test_sets,open(f'{RESULTS_DIR}/test_sets.pkl','wb'))

## 4. Run ALL methods (4 baselines + 4 variants)

In [ ]:
from railgun_plus.eval.harness import run_sweep, sweep_to_table, plot_sweep
METHODS=['expert','pibt_only','greedy','corrected',
         'variant:v2_softmax_tiebreak',
         'variant:v3_conf_gated',
         'variant:v4_prio_by_conf']
sweep=run_sweep(model,test_sets,methods=METHODS,device=device,lacam_bin=LACAM_BIN)

## 5. The decision table — does ANY variant beat pibt_only?
Look at the moderate-density rows (32, 64 agents). Whichever variant beats `pibt_only` there is the winner.

In [ ]:
import pandas as pd
rows=sweep_to_table(sweep, out_csv=f'{RESULTS_DIR}/step1_table.csv')
df=pd.DataFrame(rows)
print('=== CSR (higher=better) ==='); display(df.pivot(index='agents',columns='method',values='csr'))
print('=== Deadlock rate (lower=better) ==='); display(df.pivot(index='agents',columns='method',values='deadlock_rate'))
print('=== SoC ratio (closer to 1=better; solved only) ==='); display(df.pivot(index='agents',columns='method',values='avg_soc_ratio'))

## 6. Plot all variants on one chart

In [ ]:
plot_sweep(sweep,metric='csr',title='Step 1: corrector variants vs baselines',savepath=f'{RESULTS_DIR}/step1_csr.png')

In [ ]:
plot_sweep(sweep,metric='deadlock_rate',title='Step 1: deadlock rate',savepath=f'{RESULTS_DIR}/step1_deadlock.png')

## 7. Automated verdict
Prints a clear 'variant X beats pibt_only by Y points at density Z' summary.

In [ ]:
import numpy as np
def variant_score(method):
    s = 0.0
    for k,row in sweep[method].items():
        s += row['csr']
    return s
baseline_pibt = {k: sweep['pibt_only'][k]['csr'] for k in sweep['pibt_only']}
print('pibt_only CSRs:', {k: round(v,3) for k,v in baseline_pibt.items()})
print()
print('Variants ranked by sum of CSR across all agent counts:')
var_names = [m for m in sweep if m == 'corrected' or m.startswith('variant:')]
ranked = sorted(var_names, key=variant_score, reverse=True)
for name in ranked:
    csrs = {k: round(sweep[name][k]['csr'],3) for k in sweep[name]}
    deltas = {k: round(sweep[name][k]['csr'] - baseline_pibt[k], 3) for k in sweep[name]}
    wins = sum(1 for v in deltas.values() if v > 0)
    print(f'  {name:40s} CSR={csrs}  delta_vs_pibt={deltas}  beats_pibt_at_{wins}/{len(deltas)}_densities')

## 8. Write a summary to JOURNAL.md
This is what's important: the result of this step goes back into the repo so we always know where we are.

In [ ]:
# (Manual step — after running, append a 'Step 1 result' entry to JOURNAL.md
# in the repo with the winner and the deltas. See JOURNAL.md template.)